In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train v5 — standalone notebook

Self-contained training notebook:
- no imports from local `bev_v*.py` files;
- path resolution works with Kaggle-style dataset roots;
- low-coverage train filtering built in;
- test-matched validation split;
- calibration-aware + rover-specialist model.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, random
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

## Definitions

In [ ]:
from src.data import compute_coverage_weights
from src.geometry import resolve_info_path
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch
from src.models.decoder import SmallUNet, _UNetBlock
from src.models.v4 import build_rover_vocab
from src.models.v5 import BEVDatasetV5, MultiCamBEVv5, build_rig_features, build_specialist_vocab
from src.splits import make_test_matched_split

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def collect_low_coverage_idx(info_csv, coverage_thr=0.01):
    info_csv = Path(info_csv)
    base_dir = info_csv.parent
    info_df = pd.read_csv(info_csv, index_col=0).reset_index(drop=True)
    rows, bad_idx = [], []
    for idx, row in info_df.iterrows():
        gt_path = resolve_info_path(base_dir, row[GT_NAME])
        gt = np.load(gt_path).squeeze()
        valid = (gt != 255)
        coverage = float(valid.mean())
        pos_frac = float((gt[valid] == 1).mean()) if valid.any() else np.nan
        if coverage <= coverage_thr:
            bad_idx.append(idx)
            rows.append({
                'idx': idx,
                'rover': row.get('rover', ''),
                'ride_date': row.get('ride_date', ''),
                'message_ts': row.get('message_ts', ''),
                'coverage': coverage,
                'valid_count': int(valid.sum()),
                'pos_frac': pos_frac,
                'gt_path': str(gt_path),
            })
    bad_df = pd.DataFrame(rows).sort_values(['coverage', 'valid_count', 'rover', 'message_ts'])
    return set(bad_idx), bad_df

def _camera_pose_features(car2cam):
    cam2car = np.linalg.inv(car2cam)
    trans = cam2car[:3, 3].astype(np.float32)
    forward = cam2car[:3, 2].astype(np.float32)
    return np.concatenate([trans, forward], axis=0)

class _ResNet34Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        rn = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.proj = nn.Conv2d(128, 128, 1)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        return self.proj(x)

class _FiLM(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feat_dim * 2),
        )
    def forward(self, cond):
        gamma_beta = self.net(cond)
        return torch.chunk(gamma_beta, 2, dim=-1)


## Config

In [ ]:
cfg = {
    'run_dir': './runs/v5',
    'img_hw': (384, 704),
    'batch_size': 8,
    'grad_accum': 4,
    'epochs': 24,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'pos_weight': 5.0,
    'num_workers': 4,
    'seed': 42,
    'warmup_epochs': 4,
    'ema_decay': 0.997,
    'holdout_frac': 0.20,
    'topk_specialists': 12,
    'min_train_count': 40,
    'use_sampler': True,
    'drop_low_coverage_train': True,
    'low_coverage_thr': 0.01,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(cfg['seed'])
np.random.seed(cfg['seed'])
random.seed(cfg['seed'])

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)
print(json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))

## Rover vocab + specialists

In [ ]:
rover_vocab = build_rover_vocab(DATA_TRAIN / 'info.csv', DATA_VAL / 'info.csv', DATA_TEST / 'info.csv')
specialist_vocab = build_specialist_vocab(
    DATA_TRAIN / 'info.csv',
    DATA_TEST / 'info.csv',
    min_train_count=cfg['min_train_count'],
    topk_test_rovers=cfg['topk_specialists'],
)
print(f'rover vocab size: {len(rover_vocab)}')
print(f'specialists: {len(specialist_vocab)}')
print(json.dumps(specialist_vocab, indent=2, ensure_ascii=False))

## Split + low-coverage filter

In [ ]:
train_idx, val_idx = make_test_matched_split(
    DATA_TRAIN / 'info.csv',
    DATA_TEST / 'info.csv',
    holdout_frac=cfg['holdout_frac'],
    seed=cfg['seed'],
    cache_path=RUN_DIR / 'test_matched_split.npz',
)

bad_train_idx = set()
bad_train_df = pd.DataFrame()
if cfg['drop_low_coverage_train']:
    bad_train_idx, bad_train_df = collect_low_coverage_idx(DATA_TRAIN / 'info.csv', coverage_thr=cfg['low_coverage_thr'])
    bad_train_df.to_csv(RUN_DIR / 'bad_train_low_coverage.csv', index=False)
    with open(RUN_DIR / 'bad_train_idx.json', 'w') as f:
        json.dump(sorted(list(bad_train_idx)), f, indent=2)
    before = len(train_idx)
    train_idx = [i for i in train_idx if i not in bad_train_idx]
    print(f'low-coverage filter removed {before - len(train_idx)} train samples from matched-train split')
    print(f'global bad train samples: {len(bad_train_idx)}')
    display(bad_train_df.head(20))

print(f'train indices after filter: {len(train_idx)}')
print(f'test-matched val indices: {len(val_idx)}')

## Datasets / loaders

In [ ]:
ds_train_plain = BEVDatasetV5(DATA_TRAIN, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab, specialist_vocab=specialist_vocab)
ds_train_aug = BEVDatasetV5(DATA_TRAIN, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab, specialist_vocab=specialist_vocab)
ds_off_val = BEVDatasetV5(DATA_VAL, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab, specialist_vocab=specialist_vocab)

ds_train_warm = Subset(ds_train_plain, train_idx)
ds_train = Subset(ds_train_aug, train_idx)
ds_match_val = Subset(ds_train_plain, val_idx)

sampler = None
shuffle = True
if cfg['use_sampler']:
    weights_all = compute_coverage_weights(DATA_TRAIN / 'info.csv', cache_path=RUN_DIR / 'coverage_weights.npy')
    weights = torch.as_tensor(weights_all[train_idx], dtype=torch.double)
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(train_idx), replacement=True)
    shuffle = False

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=cfg['num_workers'], pin_memory=(device.type=='cuda'), drop_last=True)
loader_train = DataLoader(ds_train, batch_size=cfg['batch_size'], shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=cfg['num_workers'], pin_memory=(device.type=='cuda'), drop_last=True)
loader_match_val = DataLoader(ds_match_val, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
loader_off_val = DataLoader(ds_off_val, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])

sample = ds_train_plain[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)

## Model / optimizer / EMA

In [ ]:
from src.utils.training import update_ema

model = MultiCamBEVv5(num_rovers=len(rover_vocab), num_specialists=len(specialist_vocab)).to(device)
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'total params: {n_total/1e6:.2f}M')
print(f'trainable params: {n_trainable/1e6:.2f}M')

backbone_params = list(model.backbone.parameters())
other_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone']},
    {'params': other_params, 'lr': cfg['lr_head']},
], weight_decay=cfg['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight'], weight_bce=0.5, weight_dice=0.3, weight_lovasz=0.2).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad = False


## Resume

In [ ]:
start_epoch = 0
best_iou = 0.0
best_ema_iou = 0.0
if (RUN_DIR / 'last.pt').exists():
    ck = torch.load(RUN_DIR / 'last.pt', map_location=device)
    model.load_state_dict(ck['model'], strict=False)
    ema_model.load_state_dict(ck['ema'], strict=False)
    optimizer.load_state_dict(ck['opt'])
    scheduler.load_state_dict(ck['sched'])
    start_epoch = ck['epoch'] + 1
    best_iou = ck.get('best_iou', 0.0)
    best_ema_iou = ck.get('best_ema_iou', 0.0)
    print(f'resumed from epoch {start_epoch}, best={best_iou:.4f}, best_ema={best_ema_iou:.4f}')
else:
    print('fresh start')

## Eval helper

In [ ]:
@torch.no_grad()
def evaluate(m, loader, threshold=0.5):
    m.eval()
    inter, union = 0, 0
    for batch in loader:
        imgs = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rov = batch['rover_id'].to(device)
        cam_rig = batch['cam_rig'].to(device)
        global_rig = batch['global_rig'].to(device)
        specialist = batch['specialist_id'].to(device)
        gt = batch['gt'].to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = m(imgs, intr, c2c, rov, cam_rig, global_rig, specialist)
        i, u = iou_binary_batch(logits.float(), gt, threshold=threshold)
        inter += i; union += u
    return inter / max(union, 1)

## Train loop

In [ ]:
log = []
t0 = time.time()

for epoch in range(start_epoch, cfg['epochs']):
    model.train()
    train_loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    state = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f'ep{epoch:02d} [{state}]')
    for step, batch in enumerate(pbar):
        imgs = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rov = batch['rover_id'].to(device)
        cam_rig = batch['cam_rig'].to(device)
        global_rig = batch['global_rig'].to(device)
        specialist = batch['specialist_id'].to(device)
        gt = batch['gt'].to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(imgs, intr, c2c, rov, cam_rig, global_rig, specialist)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.3f}", bce=f"{parts['bce']:.2f}")

    scheduler.step()

    iou_match = evaluate(model, loader_match_val, threshold=0.5)
    iou_off = evaluate(model, loader_off_val, threshold=0.5)
    iou_match_ema = evaluate(ema_model, loader_match_val, threshold=0.5)
    iou_off_ema = evaluate(ema_model, loader_off_val, threshold=0.5)
    elapsed = (time.time() - t0) / 60

    print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | match-val: {iou_match:.4f} (ema {iou_match_ema:.4f}) | off-val: {iou_off:.4f} (ema {iou_off_ema:.4f}) | {elapsed:.1f}m")

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'iou_match': iou_match,
        'iou_off': iou_off,
        'iou_match_ema': iou_match_ema,
        'iou_off_ema': iou_off_ema,
        'minutes': elapsed,
    }
    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)

    ck = {
        'model_type': 'v5',
        'model': model.state_dict(),
        'ema': ema_model.state_dict(),
        'opt': optimizer.state_dict(),
        'sched': scheduler.state_dict(),
        'epoch': epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'cfg': cfg,
        'val_iou': iou_match,
        'val_iou_off': iou_off,
        'ema_val_iou': iou_match_ema,
        'ema_val_iou_off': iou_off_ema,
        'rover_vocab': rover_vocab,
        'specialist_vocab': specialist_vocab,
    }
    torch.save(ck, RUN_DIR / 'last.pt')
    if iou_match > best_iou:
        best_iou = iou_match
        ck['best_iou'] = best_iou
        torch.save(ck, RUN_DIR / 'best.pt')
        print(f'  new best match-val: {best_iou:.4f}')
    if iou_match_ema > best_ema_iou:
        best_ema_iou = iou_match_ema
        ck['best_ema_iou'] = best_ema_iou
        torch.save(ck, RUN_DIR / 'ema.pt')
        print(f'  new best EMA match-val: {best_ema_iou:.4f}')

print(f'\ndone in {(time.time()-t0)/60:.1f} min')
print(f'best match-val: {best_iou:.4f} | best ema: {best_ema_iou:.4f}')

## After training

Use the standalone [eval_v5.ipynb](/Users/r-shangareev/PyProjects/shine-time/ML2_2026_Competition/eval_v5.ipynb) notebook for:
- threshold sweep;
- honest EMA evaluation;
- submission generation.